# 🛍️ Visual Product Search — Local Demo Setup

## Before you run this notebook, place your output files here:

```
visual-search-demo/              ← this notebook lives here
└── outputs/
    ├── faiss_index/
    │   ├── index_*.faiss        ← download from Kaggle output
    │   └── metadata_*.json      ← download from Kaggle output
    ├── captions/
    │   └── captions_gallery.json
    ├── clip_finetuned/
    │   └── clip_finetuned_seed*.pt
    └── img/                     ← optional: DeepFashion gallery images
        └── ...                    (only needed to show result images)
```

## How to run locally

1. **Install Jupyter** if you haven't: `pip install notebook` or use VS Code with the Jupyter extension.
2. **Open this notebook** in Jupyter and run cells **top to bottom, once**.
3. Cell 1 installs all packages automatically (~2–5 min on first run).
4. After Cell 5, your app opens automatically at **http://localhost:8501**.

> **No ngrok or tunnelling needed** — the app runs directly in your browser.


## Cell 1 — Install Dependencies

In [ ]:
import sys

# Detect if a CUDA-capable GPU is available
import subprocess, shutil

def has_nvidia_gpu():
    return shutil.which("nvidia-smi") is not None

print("Installing core dependencies...")

# PyTorch — GPU build if NVIDIA detected, otherwise CPU
if has_nvidia_gpu():
    print("NVIDIA GPU detected — installing PyTorch with CUDA 11.8 support")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchvision",
        "--index-url", "https://download.pytorch.org/whl/cu118"
    ])
else:
    print("No NVIDIA GPU — installing CPU-only PyTorch (retrieval will be slower)")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchvision"
    ])

# All other packages
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics",
    "transformers>=4.30",
    "open-clip-torch",
    "accelerate",
    "streamlit",
    "faiss-cpu",       # safe on both CPU and GPU machines; switch to faiss-gpu if you prefer
    "Pillow",
    "pandas",
    "numpy",
    "scikit-learn",
    "tqdm"
])

print("\n✅ All packages installed.")

## Cell 2 — Write All Module Files

In [13]:
import os

# All files are written next to this notebook
BASE_DIR = os.getcwd()
print(f"Writing files to: {BASE_DIR}")

Writing files to: /home/sahil/Documents/Sem 6/VR/Demo


In [14]:
%%writefile config.py
"""
config.py — Local configuration for the Visual Product Search Engine.
All paths are relative to the notebook directory.
"""

import os
import torch

# ── Paths (all relative to where this notebook lives) ──────────────
BASE_DIR     = os.path.dirname(os.path.abspath(__file__))
OUTPUT_DIR   = os.path.join(BASE_DIR, "outputs")

CROPS_DIR    = os.path.join(OUTPUT_DIR, "crops")
CAPTIONS_DIR = os.path.join(OUTPUT_DIR, "captions")
INDEX_DIR    = os.path.join(OUTPUT_DIR, "faiss_index")
CLIP_FT_DIR  = os.path.join(OUTPUT_DIR, "clip_finetuned")

# IMG_DIR: where your DeepFashion gallery images live locally.
# The app still works without these — it just skips rendering gallery thumbnails.
IMG_DIR      = os.path.join(OUTPUT_DIR, "img")

# These are not used by the demo app, only kept for import compatibility
DATASET_ROOT  = OUTPUT_DIR
PARTITION_FILE = os.path.join(OUTPUT_DIR, "list_eval_partition.txt")
BBOX_FILE     = os.path.join(OUTPUT_DIR, "list_bbox_inshop.txt")
MASTER_CSV    = os.path.join(OUTPUT_DIR, "master_df.csv")
TRAIN_CSV     = os.path.join(OUTPUT_DIR, "train.csv")
QUERY_CSV     = os.path.join(OUTPUT_DIR, "query.csv")
GALLERY_CSV   = os.path.join(OUTPUT_DIR, "gallery.csv")

# Create output dirs if they don't exist
for d in [CROPS_DIR, CAPTIONS_DIR, INDEX_DIR, CLIP_FT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

# ── Models ─────────────────────────────────────────────────────────
YOLO_MODEL       = "yolov8n.pt"
BLIP2_MODEL      = "Salesforce/blip2-opt-2.7b"
CLIP_MODEL_NAME  = "ViT-B-32"
CLIP_PRETRAINED  = "openai"

# ── Hyperparameters ────────────────────────────────────────────────
CLIP_FT_LR           = 1e-5
CLIP_FT_EPOCHS       = 3
CLIP_FT_BATCH_SIZE   = 32
CLIP_FT_MARGIN       = 0.2
CLIP_FT_UNFREEZE     = 4
CLIP_FT_NUM_WORKERS  = 2

EMBEDDING_DIM        = 512
ALPHA_VALUES         = [0.7, 0.5]

HNSW_M               = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH       = 100

TOP_K_VALUES         = [5, 10, 15]
DEFAULT_TOP_K        = 10
RANDOM_SEEDS         = [2023534, 2023066, 2023612]

CAPTION_PROMPT        = "Describe this clothing item in detail including color, pattern, material, fit, and style."
CAPTION_MAX_LENGTH    = 50
ITM_RERANK_CANDIDATES = 15

Overwriting config.py


In [15]:
%%writefile utils.py
"""
utils.py — Shared utilities for the Visual Product Search Engine.
"""

import os
import re
import json
import numpy as np
from PIL import Image

def extract_item_id(image_path: str) -> str:
    parts = image_path.replace("\\", "/").split("/")
    for part in parts:
        if part.startswith("id_"):
            return part
    match = re.search(r"(id_\d+)", image_path)
    if match:
        return match.group(1)
    return "unknown_id"

def load_image(image_path: str) -> Image.Image:
    return Image.open(image_path).convert("RGB")

def save_json(data: dict, filepath: str):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)

def load_json(filepath: str) -> dict:
    with open(filepath, "r") as f:
        return json.load(f)

def normalize_embedding(emb: np.ndarray) -> np.ndarray:
    if emb.ndim == 1:
        norm = np.linalg.norm(emb)
        return emb / (norm + 1e-8)
    else:
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        return emb / (norms + 1e-8)

def fuse_embeddings(img_emb: np.ndarray, txt_emb: np.ndarray, alpha: float) -> np.ndarray:
    fused = alpha * img_emb + (1.0 - alpha) * txt_emb
    return normalize_embedding(fused)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

Overwriting utils.py


In [16]:
%%writefile yolo_detector.py
"""
yolo_detector.py — YOLO-based product localization module.
"""

import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

import config
from utils import load_image, ensure_dir

PERSON_CLASS_ID  = 0
UPPER_BODY_RATIO = (0.0, 0.55)
LOWER_BODY_RATIO = (0.40, 1.0)
FULL_BODY_RATIO  = (0.0, 1.0)

REGION_RATIOS = {
    "upper_body": UPPER_BODY_RATIO,
    "lower_body": LOWER_BODY_RATIO,
    "full_body":  FULL_BODY_RATIO,
}

class YOLODetector:
    def __init__(self, model_name=None, device=None):
        self.model_name = model_name or config.YOLO_MODEL
        self.device = device or config.DEVICE
        print(f"Loading YOLO model: {self.model_name} on {self.device}")
        self.model = YOLO(self.model_name)

    def detect(self, image_path: str, conf_threshold: float = 0.25):
        results = self.model(image_path, verbose=False, device=self.device)
        detections = []
        for result in results:
            boxes = result.boxes
            if boxes is not None and len(boxes) > 0:
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf = float(box.conf[0].cpu())
                    cls_id = int(box.cls[0].cpu())
                    if conf >= conf_threshold:
                        detections.append((x1, y1, x2, y2, conf, cls_id))
        return detections

    def _get_person_bbox(self, image_path: str, detections=None):
        if detections is None:
            detections = self.detect(image_path)
        person_dets = [d for d in detections if d[5] == PERSON_CLASS_ID]
        if person_dets:
            return max(person_dets, key=lambda d: d[4])
        elif detections:
            return max(detections, key=lambda d: (d[2] - d[0]) * (d[3] - d[1]))
        return None

    def get_body_region_crops(self, image_path: str, bbox_fallback=None, padding_ratio=0.05) -> dict:
        img = load_image(image_path)
        w, h = img.size
        detections = self.detect(image_path)
        person_det = self._get_person_bbox(image_path, detections)

        if person_det is not None:
            px1, py1, px2, py2 = person_det[:4]
        elif bbox_fallback is not None:
            px1, py1, px2, py2 = bbox_fallback
        else:
            px1, py1, px2, py2 = 0, 0, w, h

        person_h, person_w = py2 - py1, px2 - px1
        crops = {}
        for region_name, (top_ratio, bot_ratio) in REGION_RATIOS.items():
            ry1 = py1 + person_h * top_ratio
            ry2 = py1 + person_h * bot_ratio
            rx1, rx2 = px1, px2
            pad_x = person_w * padding_ratio
            pad_y = (ry2 - ry1) * padding_ratio
            rx1, ry1 = max(0, rx1 - pad_x), max(0, ry1 - pad_y)
            rx2, ry2 = min(w, rx2 + pad_x), min(h, ry2 + pad_y)
            crops[region_name] = img.crop((int(rx1), int(ry1), int(rx2), int(ry2)))

        return {"crops": crops, "detections": detections, "person_bbox": (px1, py1, px2, py2)}

    def crop_main_item(self, image_path: str, bbox_fallback=None, padding_ratio=0.05, region="full_body") -> Image.Image:
        result = self.get_body_region_crops(image_path, bbox_fallback=bbox_fallback, padding_ratio=padding_ratio)
        return result["crops"].get(region, result["crops"]["full_body"])

    def crop_batch(self, image_paths: list, bboxes=None, save_dir=None, region="full_body") -> dict:
        if save_dir:
            ensure_dir(save_dir)
        results = {}
        for path in tqdm(image_paths, desc=f"YOLO Cropping ({region})"):
            if not os.path.exists(path):
                continue
            bbox_fb = bboxes.get(path) if bboxes else None
            try:
                cropped = self.crop_main_item(path, bbox_fallback=bbox_fb, region=region)
                if save_dir:
                    flat_name = path.replace("/", "_").replace("\\", "_")
                    if not flat_name.lower().endswith((".jpg", ".png")):
                        flat_name += ".jpg"
                    save_path = os.path.join(save_dir, flat_name)
                    cropped.save(save_path)
                    results[path] = save_path
                else:
                    results[path] = cropped
            except Exception:
                try:
                    results[path] = load_image(path)
                except Exception:
                    pass
        return results

Overwriting yolo_detector.py


In [17]:
%%writefile blip2_captioner.py
"""
blip2_captioner.py — BLIP-2 captioning and Image-Text Matching module.
"""

import os
import torch
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

import config
from utils import load_image, save_json, load_json

class BLIP2Captioner:
    def __init__(self, model_name=None, device=None):
        self.model_name = model_name or config.BLIP2_MODEL
        self.device = device or config.DEVICE
        print(f"Loading BLIP-2: {self.model_name} on {self.device}")
        self.processor = Blip2Processor.from_pretrained(self.model_name)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            self.model_name,
            torch_dtype=dtype,
            device_map="auto" if self.device == "cuda" else None
        )
        if self.device == "cpu":
            self.model = self.model.to(self.device)
        self.model.eval()

    @torch.no_grad()
    def generate_caption(self, image: Image.Image, prompt=None, max_length=None) -> str:
        prompt = prompt or config.CAPTION_PROMPT
        max_length = max_length or config.CAPTION_MAX_LENGTH
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        inputs = self.processor(images=image, text=prompt, return_tensors="pt").to(self.device, dtype=dtype)
        generated_ids = self.model.generate(**inputs, max_new_tokens=max_length, num_beams=3, early_stopping=True)
        return self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    def caption_batch(self, image_paths, crop_dir=None, save_path=None):
        captions = load_json(save_path) if save_path and os.path.exists(save_path) else {}
        remaining = [p for p in image_paths if p not in captions]
        print(f"Generating captions for {len(remaining)} images...")
        for i, path in enumerate(tqdm(remaining, desc="BLIP-2 Captioning")):
            try:
                img = None
                if crop_dir:
                    flat = path.replace("/", "_").replace("\\", "_") + (".jpg" if not path.lower().endswith((".jpg", ".png")) else "")
                    cp = os.path.join(crop_dir, flat)
                    if os.path.exists(cp):
                        img = load_image(cp)
                if img is None:
                    img = load_image(path)
                captions[path] = self.generate_caption(img)
            except Exception:
                captions[path] = "clothing item"
            if save_path and (i + 1) % 100 == 0:
                save_json(captions, save_path)
        if save_path:
            save_json(captions, save_path)
        return captions

    @torch.no_grad()
    def compute_itm_score(self, image: Image.Image, text: str) -> float:
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        inputs = self.processor(images=image, text=text, return_tensors="pt").to(self.device, dtype=dtype)
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
        return -outputs.loss.item()

    def rerank_candidates(self, query_image: Image.Image, candidates: list, captions: dict) -> list:
        for cand in candidates:
            caption = captions.get(cand["image_path"], "clothing item")
            try:
                itm_score = self.compute_itm_score(query_image, caption)
            except Exception:
                itm_score = 0.0
            cand["itm_score"] = itm_score
        candidates.sort(key=lambda x: x.get("itm_score", 0), reverse=True)
        for i, cand in enumerate(candidates):
            cand["rank"] = i + 1
        return candidates

Overwriting blip2_captioner.py


In [18]:
%%writefile clip_embedder.py
"""
clip_embedder.py — CLIP embedding module.
"""

import os
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import open_clip

import config
from utils import normalize_embedding, fuse_embeddings as fuse_np, load_image

class CLIPEmbedder:
    def __init__(self, model_name=None, pretrained=None, device=None, finetuned_path=None):
        self.model_name = model_name or config.CLIP_MODEL_NAME
        self.pretrained = pretrained or config.CLIP_PRETRAINED
        self.device = device or config.DEVICE

        print(f"Loading CLIP: {self.model_name} ({self.pretrained}) on {self.device}")
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            self.model_name, pretrained=self.pretrained
        )
        self.tokenizer = open_clip.get_tokenizer(self.model_name)

        if finetuned_path and os.path.exists(finetuned_path):
            print(f"Loading fine-tuned weights from {finetuned_path}")
            state_dict = torch.load(finetuned_path, map_location=self.device)
            self.model.load_state_dict(state_dict, strict=False)

        self.model = self.model.to(self.device)
        self.model.eval()

    @torch.no_grad()
    def encode_image(self, image: Image.Image) -> np.ndarray:
        img_tensor = self.preprocess(image).unsqueeze(0).to(self.device)
        emb = self.model.encode_image(img_tensor).cpu().numpy().flatten()
        return normalize_embedding(emb)

    @torch.no_grad()
    def encode_images_batch(self, images: list, batch_size=32) -> np.ndarray:
        all_embs = []
        for i in range(0, len(images), batch_size):
            tensors = torch.stack([self.preprocess(img) for img in images[i:i+batch_size]]).to(self.device)
            all_embs.append(self.model.encode_image(tensors).cpu().numpy())
        result = np.vstack(all_embs)
        norms = np.linalg.norm(result, axis=1, keepdims=True)
        return result / (norms + 1e-8)

    @torch.no_grad()
    def encode_text(self, text: str) -> np.ndarray:
        tokens = self.tokenizer([text]).to(self.device)
        emb = self.model.encode_text(tokens).cpu().numpy().flatten()
        return normalize_embedding(emb)

    def get_fused_embedding(self, image: Image.Image, caption: str, alpha: float) -> np.ndarray:
        return fuse_np(self.encode_image(image), self.encode_text(caption), alpha)

    def extract_fused_embeddings(self, image_paths: list, captions: dict, crop_dir=None, alpha=1.0) -> dict:
        embeddings = {}
        for path in tqdm(image_paths, desc=f"Extracting Fused Embeddings (alpha={alpha})"):
            try:
                img = None
                if crop_dir:
                    flat = path.replace("/", "_").replace("\\", "_") + (".jpg" if not path.lower().endswith((".jpg", ".png")) else "")
                    cp = os.path.join(crop_dir, flat)
                    if os.path.exists(cp):
                        img = load_image(cp)
                if img is None:
                    img = load_image(path)
                img_emb = self.encode_image(img)
                embeddings[path] = fuse_np(img_emb, self.encode_text(captions.get(path, "")), alpha) if alpha < 1.0 else img_emb
            except Exception:
                embeddings[path] = np.zeros(config.EMBEDDING_DIM, dtype=np.float32)
        return embeddings

    def get_model(self): return self.model
    def get_preprocess(self): return self.preprocess
    def get_tokenizer(self): return self.tokenizer

Overwriting clip_embedder.py


In [1]:
%%writefile app.py
"""
app.py — Streamlit interactive demo for Visual Product Search.

Online Pipeline (4 explicit steps, per project spec):
  Step 1 — YOLO:     Detect product region and crop query image.
  Step 2 — CLIP:     Encode cropped query into a fused embedding.
  Step 3 — FAISS:    Retrieve top-K candidates via cosine similarity (ANN).
  Step 4 — BLIP-2:   Re-rank candidates by Image-Text Matching (ITM) score.
"""

import os
import time
import streamlit as st
import numpy as np
from PIL import Image
import faiss

import config
from utils import load_json, normalize_embedding, fuse_embeddings, load_image
from yolo_detector import YOLODetector
from clip_embedder import CLIPEmbedder

st.set_page_config(page_title="Visual Product Search", page_icon="🔍", layout="wide")
st.title("🛍️ Query-by-Image Visual Product Search Engine")
st.markdown(
    "**End-to-end pipeline:** Upload Image → "
    "**①** YOLO Crop → **Confirm** → "
    "**②** CLIP Encode → "
    "**③** FAISS Retrieve → "
    "**④** BLIP-2 Re-rank → **Results**"
)

# ── Cached resource loaders ──────────────────────────────────────────

@st.cache_resource
def load_detector():
    return YOLODetector()

@st.cache_resource
def load_clip(finetuned_path: str = ""):
    """Cache key includes finetuned_path so swapping weights reloads CLIP."""
    path = finetuned_path if finetuned_path and os.path.exists(finetuned_path) else None
    return CLIPEmbedder(finetuned_path=path)

@st.cache_resource
def load_faiss_index(index_path: str, meta_path: str):
    return faiss.read_index(index_path), load_json(meta_path)

@st.cache_resource
def load_captions():
    cap_path = os.path.join(config.CAPTIONS_DIR, "captions_gallery.json")
    return load_json(cap_path) if os.path.exists(cap_path) else {}

@st.cache_resource
def load_blip2():
    from blip2_captioner import BLIP2Captioner
    return BLIP2Captioner()

def find_available_indices():
    indices = []
    if os.path.exists(config.INDEX_DIR):
        for f in sorted(os.listdir(config.INDEX_DIR)):
            if f.startswith("index_") and f.endswith(".faiss"):
                indices.append(f.replace("index_", "").replace(".faiss", ""))
    return indices

def find_finetuned_checkpoints():
    ckpts = {"None (frozen / pre-trained)": ""}
    if os.path.exists(config.CLIP_FT_DIR):
        for f in sorted(os.listdir(config.CLIP_FT_DIR)):
            if f.endswith(".pt"):
                ckpts[f] = os.path.join(config.CLIP_FT_DIR, f)
    return ckpts

# ── Sidebar ──────────────────────────────────────────────────────────

with st.sidebar:
    st.header("⚙️ Settings")

    top_k = st.slider("Top-K results", 1, 20, config.DEFAULT_TOP_K,
                      help="Number of products to retrieve and display.")

    available_indices = find_available_indices()
    if available_indices:
        selected_index = st.selectbox(
            "Ablation config / Index",
            available_indices,
            help=(
                "Choose the FAISS index built under each ablation setting:\n"
                "  A – Vision-only CLIP (α=1)\n"
                "  B – Frozen CLIP + BLIP-2\n"
                "  C – Fine-tuned CLIP + BLIP-2"
            ),
        )
    else:
        selected_index = None
        st.error("No FAISS index found in outputs/faiss_index/")

    alpha = st.slider(
        "Alpha α (image weight in fused embedding)",
        0.0, 1.0, 0.7, 0.05,
        help="α=1 → pure visual embedding  |  α=0 → pure caption embedding\n"
             "Matches Equation 1 in the project spec: v = α·φ_V(x̂) + (1−α)·φ_T(c)"
    )

    ckpts = find_finetuned_checkpoints()
    selected_ckpt_label = st.selectbox(
        "CLIP checkpoint (fine-tuned weights)",
        list(ckpts.keys()),
        help="Select a fine-tuned CLIP checkpoint from outputs/clip_finetuned/. "
             "Choosing 'None' uses frozen pre-trained weights."
    )
    selected_ckpt_path = ckpts[selected_ckpt_label]

    use_reranking = st.checkbox(
        "Enable BLIP-2 ITM Re-ranking (Step 4)",
        value=True,
        help="Re-rank FAISS candidates using BLIP-2 Image-Text Matching. "
             "Recommended: ON for ablation configs B and C."
    )
    show_query_caption = st.checkbox(
        "Generate BLIP-2 caption for query",
        value=True,
        help="Show the BLIP-2-generated caption for your query image (for inspection)."
    )

    st.markdown("---")
    st.caption(f"Device: `{config.DEVICE}`")
    st.markdown("""
**Ablation legend**
| Config | Description |
|--------|-------------|
| A | Vision-only CLIP, α=1 |
| B | Frozen CLIP + BLIP-2 |
| C | Fine-tuned CLIP + BLIP-2 |
""")

# ── Image Upload ──────────────────────────────────────────────────────

uploaded_file = st.file_uploader(
    "📸 Upload a product query image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:
    query_image = Image.open(uploaded_file).convert("RGB")

    # Reset pipeline state on new upload
    current_file_id = uploaded_file.name + str(uploaded_file.size)
    if st.session_state.get("_file_id") != current_file_id:
        st.session_state._file_id = current_file_id
        st.session_state.confirmed = None       # None | "confirmed" | "recrop"
        st.session_state.query_caption = None

    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    temp_path = os.path.join(config.OUTPUT_DIR, "temp_query.jpg")
    query_image.save(temp_path)

    # ════════════════════════════════════════════════════════════════
    # STEP 1 — YOLO: Product Localization
    # ════════════════════════════════════════════════════════════════
    st.markdown("---")
    st.subheader("① Product Localization  —  YOLO")
    st.caption(
        "YOLO detects the primary clothing item and crops the image to reduce "
        "background noise before encoding."
    )

    with st.spinner("Running YOLO detection..."):
        detector = load_detector()
        region_result = detector.get_body_region_crops(temp_path)
        crops = region_result["crops"]
        detections = region_result["detections"]

    col_orig, col_crop, col_ctrl = st.columns([1, 1, 2])

    with col_orig:
        st.image(query_image, caption="Original upload", use_container_width=True)
        st.caption(f"Objects detected by YOLO: **{len(detections)}**")

    with col_crop:
        selected_region = st.radio(
            "Crop region:",
            ["full_body", "upper_body", "lower_body"],
            format_func=lambda x: x.replace("_", " ").title(),
        )
        st.image(
            crops[selected_region],
            caption=f"YOLO crop — {selected_region.replace('_', ' ').title()}",
            use_container_width=True,
        )

    with col_ctrl:
        st.markdown("#### Confirmation Required")
        st.info(
            "Inspect the YOLO crop. "
            "**Confirm** to use it as the query, or **Re-crop** to fall back to the "
            "full original image."
        )
        b1, b2 = st.columns(2)
        with b1:
            if st.button("Confirm Crop & Proceed", use_container_width=True, type="primary"):
                st.session_state.confirmed = "confirmed"
                st.session_state.query_caption = None   # reset caption for fresh crop
        with b2:
            if st.button("Re-crop (Use Full Image)", use_container_width=True):
                st.session_state.confirmed = "recrop"
                st.session_state.query_caption = None

    if st.session_state.confirmed == "confirmed":
        st.success(f"Using YOLO crop — region: **{selected_region.replace('_', ' ').title()}**")
    elif st.session_state.confirmed == "recrop":
        st.warning("Using full original image (YOLO crop bypassed)")

    # ════════════════════════════════════════════════════════════════
    # STEPS 2–4: Run only after user confirms
    # ════════════════════════════════════════════════════════════════
    if st.session_state.confirmed and selected_index:

        target = (
            crops[selected_region]
            if st.session_state.confirmed == "confirmed"
            else query_image
        )

        # ────────────────────────────────────────────────────────────
        # STEP 2 — CLIP: Query Encoding
        # ────────────────────────────────────────────────────────────
        st.markdown("---")
        st.subheader("② Query Encoding  —  CLIP")
        st.caption(
            f"The cropped image is embedded by CLIP (ViT-B/32). "
            f"When α < 1, a BLIP-2 caption is used to produce the fused embedding "
            f"v = α·φ_V(x̂) + (1−α)·φ_T(c).  "
            f"Current **α = {alpha:.2f}** (checkpoint: *{selected_ckpt_label}*)."
        )

        with st.spinner("Loading CLIP and encoding query image..."):
            t0 = time.time()
            clip = load_clip(selected_ckpt_path)
            img_emb = clip.encode_image(target)

            # Optionally fuse with a caption embedding when α < 1
            if alpha < 1.0 and show_query_caption:
                try:
                    blip2 = load_blip2()
                    if st.session_state.query_caption is None:
                        st.session_state.query_caption = blip2.generate_caption(target)
                    query_emb = fuse_embeddings(img_emb, clip.encode_text(st.session_state.query_caption), alpha)
                except Exception:
                    query_emb = img_emb   # fall back to image-only
            else:
                query_emb = img_emb

            query_emb_2d = query_emb.reshape(1, -1).astype(np.float32)
            elapsed_clip = time.time() - t0

        # Show embedding preview
        preview = query_emb[:16]
        st.code(
            f"Embedding (first 16 / {config.EMBEDDING_DIM} dims):\n"
            f"[{', '.join(f'{v:.4f}' for v in preview)}, ...]",
            language=None,
        )
        st.success(
            f"Query embedded  |  shape: {query_emb_2d.shape}  |  "
            f"L2 norm: {float(np.linalg.norm(query_emb_2d)):.4f}  |  "
            f"time: {elapsed_clip*1000:.0f} ms"
        )

        # Optionally generate + display BLIP-2 query caption
        if show_query_caption:
            if st.session_state.query_caption is None:
                with st.spinner("Generating BLIP-2 caption for query image..."):
                    try:
                        blip2 = load_blip2()
                        st.session_state.query_caption = blip2.generate_caption(target)
                    except Exception as e:
                        st.session_state.query_caption = f"(unavailable — {e})"
            st.markdown(
                f"**BLIP-2 query caption:** *{st.session_state.query_caption}*"
            )

        # ────────────────────────────────────────────────────────────
        # STEP 3 — FAISS: Candidate Retrieval
        # ────────────────────────────────────────────────────────────
        st.markdown("---")
        st.subheader("③ Candidate Retrieval  —  FAISS / HNSW")
        st.caption(
            "The query embedding is compared against all gallery embeddings "
            "using cosine similarity (inner product on L2-normalised vectors) "
            "via an Approximate Nearest Neighbour index."
        )

        with st.spinner("Searching FAISS index..."):
            t0 = time.time()
            idx_path = os.path.join(config.INDEX_DIR, f"index_{selected_index}.faiss")
            mt_path  = os.path.join(config.INDEX_DIR, f"metadata_{selected_index}.json")
            index, metadata = load_faiss_index(idx_path, mt_path)

            # Fetch extra candidates when re-ranking will follow
            search_k = min(top_k * 3 if use_reranking else top_k, index.ntotal)
            distances, faiss_indices = index.search(query_emb_2d, search_k)
            elapsed_faiss = time.time() - t0

        candidates = [
            {
                "image_path": metadata[idx]["image_path"],
                "item_id":    metadata[idx]["item_id"],
                "score":      float(dist),
                "rank":       r + 1,
                "itm_score":  None,
            }
            for r, (dist, idx) in enumerate(zip(distances[0], faiss_indices[0]))
            if 0 <= idx < len(metadata)
        ]

        c1, c2, c3 = st.columns(3)
        c1.metric("Gallery size", f"{index.ntotal:,}")
        c2.metric("Candidates fetched", len(candidates))
        c3.metric("Search time", f"{elapsed_faiss*1000:.0f} ms")
        st.success(f"Top-{len(candidates)} candidates retrieved (will show top-{top_k} after re-ranking)")

        # ────────────────────────────────────────────────────────────
        # STEP 4 — BLIP-2: ITM Re-ranking
        # ────────────────────────────────────────────────────────────
        st.markdown("---")
        if use_reranking:
            st.subheader("④ Semantic Re-ranking  —  BLIP-2 ITM")
            st.caption(
                "A BLIP-2 Image-Text Matching score is computed between the query image "
                "and each candidate's pre-computed caption. Candidates are re-ranked by "
                "ITM score to boost semantic precision."
            )
            with st.spinner(f"Re-ranking {len(candidates)} candidates via BLIP-2 ITM..."):
                t0 = time.time()
                try:
                    blip2 = load_blip2()
                    captions = load_captions()
                    candidates = blip2.rerank_candidates(target, candidates, captions)
                    elapsed_itm = time.time() - t0
                    st.success(
                        f"Re-ranked {len(candidates)} candidates by ITM score  |  "
                        f"time: {elapsed_itm*1000:.0f} ms"
                    )
                except Exception as e:
                    st.warning(f"Re-ranking failed ({e}) — displaying FAISS-ranked results.")
        else:
            st.subheader("④ Semantic Re-ranking  —  BLIP-2 ITM  *(disabled)*")
            st.caption("Enable **BLIP-2 ITM Re-ranking** in the sidebar to activate Step 4.")

        # ════════════════════════════════════════════════════════════
        # Results grid
        # ════════════════════════════════════════════════════════════
        st.markdown("---")
        results = candidates[:top_k]
        st.subheader(f"Top-{len(results)} Retrieved Products")

        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Results shown", len(results))
        m2.metric("Best similarity", f"{results[0]['score']:.4f}" if results else "—")
        m3.metric("Re-ranking", "ON" if use_reranking else "OFF")
        m4.metric("α (image weight)", f"{alpha:.2f}")

        cols_per_row = 5
        for row_start in range(0, len(results), cols_per_row):
            cols = st.columns(cols_per_row)
            for j, col in enumerate(cols):
                idx = row_start + j
                if idx >= len(results):
                    break
                r = results[idx]
                with col:
                    if os.path.exists(r["image_path"]):
                        st.image(load_image(r["image_path"]), use_container_width=True)
                    else:
                        st.markdown(
                            "<div style='background:#f0f0f0;height:160px;display:flex;"
                            "align-items:center;justify-content:center;border-radius:6px;'>"
                            "🖼️<br><small>not available</small></div>",
                            unsafe_allow_html=True,
                        )
                    st.markdown(f"**Rank {r['rank']}**")
                    st.caption(f"`{r['item_id']}`")
                    st.caption(f"Sim: `{r['score']:.4f}`")
                    if r.get("itm_score") is not None:
                        st.caption(f"ITM: `{r['itm_score']:.4f}`")

    elif st.session_state.confirmed and not selected_index:
        st.error(
            "No FAISS index found. "
            "Place your `index_*.faiss` and `metadata_*.json` files inside "
            "`outputs/faiss_index/` and re-run Cell 4."
        )

    # Cleanup temp file
    if os.path.exists(temp_path):
        try:
            os.remove(temp_path)
        except Exception:
            pass

else:
    st.markdown("---")
    st.info("Upload a clothing image above to start the pipeline.")
    st.markdown("""
**How the pipeline works once you upload:**

| Step | Module | What happens |
|------|--------|--------------|
| ① | **YOLO** | Detects clothing region; crops background noise |
| ✅ | **You** | Confirm crop or choose full image |
| ② | **CLIP** | Encodes query into a 512-d fused embedding (Eq. 1) |
| ③ | **FAISS** | ANN search returns top-K candidates via cosine similarity |
| ④ | **BLIP-2 ITM** | Re-ranks candidates using image–text matching score |
""")


Writing app.py


## Cell 3 — Fix Metadata Image Paths

Your `metadata_*.json` files contain Kaggle-absolute paths like `/kaggle/input/.../img/...`.
This cell rewrites them to point to your local `outputs/img/` folder.

- **If you have the gallery images locally** → set `LOCAL_IMG_DIR` below and run.
- **If you don't have gallery images** → run anyway; the app will just show a placeholder instead of the image thumbnail.

In [20]:
import os, json, glob

INDEX_DIR     = os.path.join(os.getcwd(), "outputs", "faiss_index")
LOCAL_IMG_DIR = os.path.join(os.getcwd(), "outputs", "img")   # change this if your images are elsewhere

meta_files = glob.glob(os.path.join(INDEX_DIR, "metadata_*.json"))
print(f"Found {len(meta_files)} metadata file(s) to patch:\n")

for meta_path in meta_files:
    with open(meta_path, "r") as f:
        metadata = json.load(f)

    patched = 0
    for entry in metadata:
        old_path = entry.get("image_path", "")
        # Extract relative portion after 'img/' — works for both Kaggle and other absolute paths
        if "img/" in old_path:
            rel = old_path.split("img/", 1)[-1]
            entry["image_path"] = os.path.join(LOCAL_IMG_DIR, rel)
            patched += 1

    with open(meta_path, "w") as f:
        json.dump(metadata, f)

    print(f"  ✅ {os.path.basename(meta_path)} — patched {patched}/{len(metadata)} entries")

print("\nDone. Run Cell 4 to launch the app.")

Found 9 metadata file(s) to patch:

  ✅ metadata_alpha0.5_configC_seed2023612.json — patched 12612/12612 entries
  ✅ metadata_alpha0.7_configC_seed2023066.json — patched 12612/12612 entries
  ✅ metadata_alpha0.7_configC_seed2023612.json — patched 12612/12612 entries
  ✅ metadata_alpha0.7_configC_seed2023534.json — patched 12612/12612 entries
  ✅ metadata_alpha0.7_configB.json — patched 12612/12612 entries
  ✅ metadata_alpha1.0_configA.json — patched 12612/12612 entries
  ✅ metadata_alpha0.5_configC_seed2023534.json — patched 12612/12612 entries
  ✅ metadata_alpha0.5_configC_seed2023066.json — patched 12612/12612 entries
  ✅ metadata_alpha0.5_configB.json — patched 12612/12612 entries

Done. Run Cell 4 to launch the app.


## Cell 4 — Verify Everything Is In Place

In [21]:
import os, glob

checks = {
    "config.py":         os.path.exists("config.py"),
    "utils.py":          os.path.exists("utils.py"),
    "yolo_detector.py":  os.path.exists("yolo_detector.py"),
    "blip2_captioner.py":os.path.exists("blip2_captioner.py"),
    "clip_embedder.py":  os.path.exists("clip_embedder.py"),
    "app.py":            os.path.exists("app.py"),
    "FAISS index files": len(glob.glob("outputs/faiss_index/index_*.faiss")) > 0,
    "Metadata JSON files":len(glob.glob("outputs/faiss_index/metadata_*.json")) > 0,
    "Captions JSON":     os.path.exists("outputs/captions/captions_gallery.json"),
}

all_ok = True
for name, status in checks.items():
    icon = "✅" if status else "❌"
    print(f"  {icon}  {name}")
    if not status:
        all_ok = False

print()
if all_ok:
    print("Everything looks good! Proceed to Cell 5.")
else:
    print("⚠️  Fix the missing items above before launching the app.")

  ✅  config.py
  ✅  utils.py
  ✅  yolo_detector.py
  ✅  blip2_captioner.py
  ✅  clip_embedder.py
  ✅  app.py
  ✅  FAISS index files
  ✅  Metadata JSON files
  ✅  Captions JSON

Everything looks good! Proceed to Cell 5.


## Cell 5 — Launch the Streamlit App

Run this cell. Your browser should open automatically at **http://localhost:8501**.

If the browser doesn't open, navigate there manually.

The cell keeps running while the app is alive — that's expected. **Interrupt the kernel** or run Cell 6 to stop it.


In [11]:
import subprocess, sys, os, time

# Write streamlit config to skip email prompt
config_dir = os.path.expanduser("~/.streamlit")
os.makedirs(config_dir, exist_ok=True)
with open(os.path.join(config_dir, "config.toml"), "w") as f:
    f.write("[browser]\ngatherUsageStats = false\n")

# Also set via environment variable as backup
env = os.environ.copy()
env["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"

# Kill any previous instance
subprocess.call(["pkill", "-f", "streamlit"], stderr=subprocess.DEVNULL)
time.sleep(1)

print("Starting Streamlit app...")
print("Open → http://localhost:8501")
print("Interrupt this cell to stop the server.\n")

subprocess.run(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "false",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    cwd=os.getcwd(),
    env=env
)

Starting Streamlit app...
Open → http://localhost:8501
Interrupt this cell to stop the server.



2026-05-14 16:59:28.261 Uvicorn server started on 0.0.0.0:8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.16.0.2:8501



[transformers] Accessing `__path__` from `.models.aria.image_processing_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.aria.image_processing_pil_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.auto.image_processing_auto`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.beit.image_processing_beit`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.beit.image_processing_pil_beit`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.bit.image_pr

Loading YOLO model: yolov8n.pt on cpu


...

Loading BLIP-2: Salesforce/blip2-opt-2.7b on cpu


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]/home/sahil/Documents/Sem 6/VR/Demo/.venv/lib/python3.14/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 9996.33 MB. The target location /home/sahil/.cache/huggingface/hub/models--Salesforce--blip2-opt-2.7b/blobs only has 4955.48 MB free disk space.
  warnings.warn(
/home/sahil/Documents/Sem 6/VR/Demo/.venv/lib/python3.14/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 4982.88 MB. The target location /home/sahil/.cache/huggingface/hub/models--Salesforce--blip2-opt-2.7b/blobs only has 4955.47 MB free disk space.
  warnings.warn(


  Stopping...


Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

## Cell 6 (Optional) — Stop the App

Run this if you want to cleanly stop the server without restarting the kernel.

In [12]:
import subprocess

subprocess.call(["pkill", "-f", "streamlit"])
print("Streamlit server stopped.")


Streamlit server stopped.
